# 03 · Matriz ABC × XYZ sobre train

Solo carga días ≤ fin de train y no lee `reports/eda_five_years/`. Antes de calcular se declaran la regla ABC, la búsqueda de cortes del CV, `zero_alpha` y `long_run`. Se congelan una vez (Principio V): `zero_thresholds`, `train_cuts` y `matrix_train`.

Orden: (1) umbral de racha de ceros por artículo (FR-032) y rachas en NaN para el CV; (2) cortes ABC con `select_abc_cuts` y umbrales XYZ con `find_cv_cuts` sobre train (FR-013); (3) matriz global.

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

In [ ]:
from src.segmentation import run_train_matrix

res = run_train_matrix(cfg)
matrix = res["matrix"]
pd.Series(res["cuts"], name="elegido en train")

### Umbral de racha de ceros (FR-032)

Para cada artículo, k = menor largo con P(racha más larga ≥ k) ≤ α bajo días independientes Bernoulli(p₀), con n y p₀ medidos entre la primera y la última venta de train. Las rachas internas ≥ k, el arranque y los ceros finales pasan a NaN en el CV. Wald–Wolfowitz es solo diagnóstico (z > 0 = ceros más agrupados que el azar).

**Límites:** p₀ incluye las rachas que se quieren detectar, así que el umbral es conservador; con p₀ bajo el umbral es corto (≈ 3 días con p₀ = 0,01). No hay etiqueta de quiebre para validar el método.

In [ ]:
zt = res["zero_thresholds"]
alpha = cfg["segmentation"]["zero_alpha"]
display(zt.groupby(["source", "reason"], dropna=False).size().rename("artículos").to_frame())
est = zt[zt["source"] == "estimated"]
display(est["threshold"].describe().to_frame("umbral (días)").T)
clustered = (est["ww_pvalue"] < alpha) & (est["ww_z"] > 0)
print(f"ceros agrupados más que por azar (Wald–Wolfowitz, p < {alpha}): {clustered.mean():.1%} de {len(est)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
est["threshold"].clip(upper=40).plot.hist(bins=40, ax=axes[0], color="0.4")
axes[0].set(title="Umbral por artículo (recortado en 40)", xlabel="días")
axes[1].scatter(est["p0"], est["threshold"], s=6, color="0.3")
axes[1].set(title="Umbral frente a la tasa de ceros", xlabel="p₀", ylabel="umbral (días)", yscale="log")
plt.tight_layout()
plt.show()

### Cortes de train

ABC: escenarios de `select_abc_cuts` ordenados por violación de la regla (los 5 mejores). XYZ: segmentos de `find_cv_cuts` sobre el CV enmascarado.

In [ ]:
display(res["abc_scenarios"].head(5))
res["cv_segments"]

### Crudo frente a enmascarado

Clase XYZ con los mismos umbrales elegidos, calculada con el CV crudo (sin enmascarar) y con el enmascarado.

In [ ]:
from src.segmentation import classify_xyz

m = matrix.set_index("item_id")
crude = classify_xyz(res["cv_raw"].reindex(m.index), res["cuts"]["xyz_thresholds"])
pd.crosstab(crude.fillna("-").rename("crudo"), m["xyz_class"].fillna("-").rename("enmascarado"), margins=True)

Matriz global de train.

In [ ]:
pd.crosstab(matrix["abc_class"], matrix["xyz_class"].fillna("-"), margins=True)

Cobertura (SC-001): cada artículo activo tiene clase ABC y XYZ, o motivo de exclusión.

In [ ]:
active = matrix[matrix["exclusion_reason"] != "inactive_in_set"]
ok = active["abc_class"].notna() & (active["xyz_class"].notna() | active["exclusion_reason"].notna())
print(f"cobertura: {ok.mean():.1%} de {len(active)} artículos activos")
assert ok.all()
matrix["exclusion_reason"].value_counts(dropna=False)

Sensibilidad de los valores por defecto (±5 pp, FR-013), solo sobre train.

In [ ]:
res["sensitivity"].pivot_table(index=["dimension", "variant"], columns="class", values="count")

Verificación manual de 10 artículos (SC-003): el gasto se recalcula desde unidades × precio por tienda-día, y el CV desde bloques de 28 días con las rachas de ceros en NaN según el umbral congelado, sin usar `build_matrix`.

In [ ]:
from src.config import train_end, train_years
from src.data import load_interim
from src.segmentation import mask_zero_runs

thr = zt.set_index("item_id")["threshold"]

end = train_end(cfg)
start = min(a for a, _ in train_years(cfg).values())
raw = load_interim(cfg, end, series_level=True)
si = raw["series_index"]
ranked = active.reset_index(drop=True)
sample = pd.concat([ranked.head(5), ranked.sample(min(5, len(ranked)), random_state=0)]).drop_duplicates("item_id")
total = ranked["spend"].sum()
checks = []
for _, row in sample.iterrows():
    rows = si.index[si["item_id"] == row["item_id"]]
    u = np.asarray(raw["units"][rows, start - 1:end], dtype=float)
    p = np.asarray(raw["prices_daily"][rows, start - 1:end], dtype=float)
    spend = np.nansum(u * p)
    cum_before = ranked.loc[ranked["spend"] > row["spend"], "spend"].sum() + ranked.loc[(ranked["spend"] == row["spend"]) & (ranked["item_id"] < row["item_id"]), "spend"].sum()
    daily = np.where(np.isnan(u).all(axis=0), np.nan, np.nansum(u, axis=0))
    full = np.full((1, end), np.nan)
    full[0, start - 1:end] = daily
    daily = mask_zero_runs(full, [row["item_id"]], thr, start, end)[0, start - 1:end]
    blocks = daily[: len(daily) // 28 * 28].reshape(-1, 28)
    sums = blocks[~np.isnan(blocks).any(axis=1)].sum(axis=1)
    cv = sums.std(ddof=0) / sums.mean() * 100 if len(sums) >= cfg["segmentation"]["cv_min_blocks"] and sums.mean() > 0 else np.nan
    checks.append({"item_id": row["item_id"], "spend_ok": np.isclose(spend, row["spend"]),
                   "cum_pct_ok": np.isclose(cum_before / total, row["cum_pct_before"]),
                   "cv_ok": (np.isnan(cv) and np.isnan(row["cv"])) or np.isclose(cv, row["cv"]),
                   "abc_class": row["abc_class"], "xyz_class": row["xyz_class"]})
checks = pd.DataFrame(checks)
assert checks[["spend_ok", "cum_pct_ok", "cv_ok"]].all().all()
checks